# 범용 보안 AI 학습 — DiverseVul (실 CVE, 150+ CWE) on Colab GPU

원격 샌드박스는 Google Drive/HuggingFace가 막혀 DiverseVul을 못 받지만 **Colab에선 됩니다.**
이 노트북은 정직하게:
- **DiverseVul**(실제 CVE 수정커밋 기반, 함수 33만개 / 취약 1.8만, 150+ CWE)를 받고
- 중복 제거 + **프로젝트 단위 홀드아웃**(유출 차단)으로
- ① TF-IDF+LightGBM(빠름·CPU·100% 내꺼) ② CodeBERT 파인튜닝(GPU·SOTA) 둘 다 학습
- **PR-AUC/recall** 로 평가(불균형이라 정확도는 함정)

**런타임 → 런타임 유형 변경 → GPU(T4)** 로 먼저 바꾸세요.

## 1) 설치

In [ ]:
!pip -q install gdown lightgbm scikit-learn transformers datasets torch --upgrade
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음(런타임을 GPU로 바꾸세요)')

## 2) DiverseVul 다운로드
공식 배포(Google Drive). 파일 id는 프로젝트 README 기준이며, 바뀌면
https://github.com/wagner-group/diversevul 에서 최신 링크로 교체하세요.

In [ ]:
import gdown, os, zipfile, glob
# DiverseVul 데이터셋 (README의 다운로드 링크 파일 id)
FILE_ID = '12IWKhmLhq7qn5B_iXgn5YerOQtkH-6RG'
raw = 'diversevul_download'
if not os.path.exists(raw):
    gdown.download(id=FILE_ID, output=raw, quiet=False)
print('받은 크기:', round(os.path.getsize(raw)/1e6, 1), 'MB')
# zip이면 자동 압축해제 후 안의 json을 찾고, 아니면 그대로 사용
if zipfile.is_zipfile(raw):
    zipfile.ZipFile(raw).extractall('diversevul_extracted')
    cands = glob.glob('diversevul_extracted/**/*.json', recursive=True)
    DATA_PATH = max(cands, key=os.path.getsize)
else:
    DATA_PATH = raw
print('사용할 데이터 파일:', DATA_PATH)

## 3) 로드 + 중복제거 + CWE 분포
DiverseVul은 JSON-lines(한 줄당 함수 하나). 필드: func, target(0/1), cwe[], project, commit_id.

In [ ]:
import json, re, hashlib, numpy as np, collections

def load_diversevul(path):
    recs = []
    with open(path, encoding='utf-8', errors='ignore') as f:
        head = f.read(1)
        f.seek(0)
        if head == '[':                      # JSON array
            recs = json.load(f)
        else:                                # JSON lines
            for line in f:
                line = line.strip()
                if line:
                    try: recs.append(json.loads(line))
                    except Exception: pass
    return recs

recs = load_diversevul(DATA_PATH)   # 앞 셀에서 정한 파일(zip이면 압축해제된 json)
print('원본 레코드:', len(recs))

seen=set(); codes=[]; y=[]; groups=[]; cwes=[]
for r in recs:
    code = r.get('func') or r.get('code') or ''
    if len(code) < 20: continue
    k = hashlib.md5(re.sub(r'\s+',' ',code).strip().encode()).hexdigest()
    if k in seen: continue
    seen.add(k)
    codes.append(code)
    y.append(int(r.get('target', 0)))
    groups.append(str(r.get('project') or r.get('commit_id') or 'na'))
    cwes.append(tuple(r.get('cwe', [])))
y = np.array(y); groups = np.array(groups)
print(f'중복제거 {len(codes)} | 취약 {int(y.sum())} ({y.mean()*100:.1f}%) | 프로젝트 {len(set(groups))}개')
cwe_flat = collections.Counter(c for t in cwes for c in t)
print('고유 CWE:', len(cwe_flat), '| 상위10:', cwe_flat.most_common(10))

## 4) 베이스라인 — TF-IDF + LightGBM (빠름, CPU, 사전학습 0 = 100% 내꺼)
프로젝트 단위 홀드아웃. DiverseVul은 실전형(취약 6%)이라 PR-AUC/recall 을 봅니다.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from lightgbm import LGBMClassifier

vec = TfidfVectorizer(token_pattern=r'[A-Za-z_]\w+|[^\sA-Za-z0-9]',
                      ngram_range=(1,2), min_df=3, max_features=40000)
X = vec.fit_transform(codes)
pos_w = (y==0).sum()/max((y==1).sum(),1)
gkf = GroupKFold(n_splits=5); aucs=[]; aps=[]; ya=[]; pa=[]
for tr,te in gkf.split(X,y,groups):
    m = LGBMClassifier(n_estimators=500, learning_rate=0.05, num_leaves=127,
                       scale_pos_weight=pos_w, random_state=42, n_jobs=-1, verbose=-1)
    m.fit(X[tr], y[tr]); p = m.predict_proba(X[te])[:,1]
    aucs.append(roc_auc_score(y[te],p)); aps.append(average_precision_score(y[te],p))
    ya += list(y[te]); pa += list((p>0.5).astype(int))
print(f'ROC-AUC {np.mean(aucs):.3f} | PR-AUC {np.mean(aps):.3f} (취약률 {y.mean()*100:.1f}%)')
print(classification_report(ya, pa, target_names=['safe','vuln'], digits=3))

## 5) SOTA — CodeBERT 파인튜닝 (GPU 필요)
**주의(정직):** CodeBERT는 Microsoft가 사전학습한 모델이라 '온전히 내꺼'는 아닙니다
(MS 뇌 + 내 파인튜닝). 대신 소량 데이터로도 성능이 높습니다. 100% 자작을 원하면
6번(밑바닥 신경망)을 쓰세요. 시간 절약을 위해 SAMPLE 로 일부만 학습합니다.

In [ ]:
import torch, numpy as np
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, average_precision_score

SAMPLE = 40000  # GPU 시간 절약(전체 쓰려면 None). T4에서 4만개 3에폭 ≈ 20~40분
idx = np.arange(len(codes))
if SAMPLE and len(idx) > SAMPLE:
    rng = np.random.default_rng(42); idx = rng.choice(idx, SAMPLE, replace=False)
sub_codes=[codes[i] for i in idx]; sub_y=y[idx]; sub_g=groups[idx]
tr, te = next(GroupShuffleSplit(1, test_size=0.2, random_state=42).split(sub_codes, sub_y, sub_g))

tok = AutoTokenizer.from_pretrained('microsoft/codebert-base')
def enc(texts): return tok(list(texts), truncation=True, padding='max_length', max_length=256)
class DS(torch.utils.data.Dataset):
    def __init__(self, ids, labels): self.e=enc([sub_codes[i] for i in ids]); self.l=[int(sub_y[i]) for i in ids]
    def __len__(self): return len(self.l)
    def __getitem__(self,i): return {**{k:torch.tensor(v[i]) for k,v in self.e.items()}, 'labels':torch.tensor(self.l[i])}

model = AutoModelForSequenceClassification.from_pretrained('microsoft/codebert-base', num_labels=2)
args = TrainingArguments('out', per_device_train_batch_size=16, per_device_eval_batch_size=32,
                         num_train_epochs=3, learning_rate=2e-5, fp16=torch.cuda.is_available(),
                         logging_steps=50, report_to='none', save_strategy='no')
trainer = Trainer(model, args, train_dataset=DS(tr,None), eval_dataset=DS(te,None))
trainer.train()

pred = trainer.predict(DS(te,None))
prob = torch.softmax(torch.tensor(pred.predictions),1)[:,1].numpy()
yte = np.array([int(sub_y[i]) for i in te])
print(f'CodeBERT ROC-AUC {roc_auc_score(yte,prob):.3f} | PR-AUC {average_precision_score(yte,prob):.3f}')

## 6) (선택) 밑바닥부터 신경망 — 100% 내꺼 (사전학습 0)
CodeBERT 없이 random init 부터 학습. 데이터가 충분하면(DiverseVul 33만) 여기서도 학습됩니다.

In [ ]:
# 저장소의 train_from_scratch_nn.py 의 TinyVulnNet 을 이 데이터에 적용하면 됩니다.
# (토크나이저·임베딩·인코더·분류기 전부 random init — 남의 가중치 0)
print('train_from_scratch_nn.py 참고: 사전학습 없는 CNN 분류기. 데이터가 크면 성능이 오릅니다.')

## 결론(정직)
- DiverseVul은 실전형(취약 소수)이라 VulDeePecker(0.96)보다 **낮게 나오는 게 정상**입니다.
  실제 프로젝트의 미묘한 취약점이라 어렵습니다(Devign이 0.52였던 것과 같은 이유).
- **CodeBERT > TF-IDF > 밑바닥** 순으로 보통 성능이 나오지만, CodeBERT는 MS 사전학습을 빌린 것.
- 더 올리려면: PrimeVul 추가, GraphCodeBERT(데이터흐름), 여러 CWE 균형, 에폭·max_length 확대.